In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# 自动向上查找项目根目录 (含 .gitignore 的文件夹)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# 切换 cwd 到项目根, 使所有相对路径 (Stage1_Exploration/, Refined_Results_v4/ 等) 保持有效
os.chdir(PROJECT_ROOT)
# 让 notebooks 能 `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from pysr import PySRRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

# ==========================================
# 0. 实验参数配置 (Experiment Configuration)
# ==========================================
# 定义不同的训练集大小，用于评估模型的数据效率
# 从极小样本 (100) 到全量样本 (44400)
DATA_SIZES = [
    100, 200, 300, 400, 500, 
    750, 1000, 2000, 3000, 4000, 
    5000, 7500, 10000, 20000, 30000, 44400
]

FIXED_NOISE = 0.5  # 固定噪声水平 (50%)
N_REPEATS = 20     # 每个样本量重复实验次数，以获取统计显著性
SCALE = 1e7        # 浓度缩放因子
FEATURE_NAMES = ['V_in', 'C_in', 'Area', 'Distance']

OUTPUT_DIR = "Data_Efficiency_Results"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ==========================================
# 1. 数据准备 (Data Preparation)
# ==========================================
print("正在加载数据...")
df = pd.read_csv('data/train_dataset_ready.csv')

# 缩放处理
df['C_in'] = df['C_in'] * SCALE
df['C_out'] = df['C_out'] * SCALE

# 划分固定的测试集 (Test Set)
# 关键：测试集必须在所有实验中保持一致，以确保比较的公平性
unique_cases = df['Case'].unique()
train_cases_pool, test_cases_fixed = train_test_split(unique_cases, test_size=0.2, random_state=42)

# 准备测试数据 (保持干净，作为 Ground Truth)
test_df = df[df['Case'].isin(test_cases_fixed)].copy()
X_test = test_df[FEATURE_NAMES].values
y_test_clean = test_df['C_out'].values

# 准备训练数据池 (Training Pool)
# 后续将从这里根据 DATA_SIZES 进行不同规模的采样
train_df_pool = df[df['Case'].isin(train_cases_pool)].copy()
train_pool_indices = np.arange(len(train_df_pool))

print(f"数据准备完毕。实验将重复 {N_REPEATS} 次。")

# ==========================================
# 2. 循环测试 (Loop Experiment)
# ==========================================
results_summary = []

for size in DATA_SIZES:
    print(f"\n{'='*40}")
    print(f"正在评估训练集大小: {size}")
    print(f"{'='*40}")
    
    current_size = min(size, len(train_pool_indices))
    r2_scores_hybrid = []
    
    for i in range(N_REPEATS):
        # 设置不同的随机种子，确保每次采样的样本不同
        seed = 42 + size + i 
        np.random.seed(seed)
        
        # A. 随机采样 (Random Sampling)
        sampled_indices = np.random.choice(train_pool_indices, current_size, replace=False)
        X_train_sub = train_df_pool.iloc[sampled_indices][FEATURE_NAMES].values
        y_train_clean_sub = train_df_pool.iloc[sampled_indices]['C_out'].values
        
        # B. 注入噪声 (Inject Noise)
        # 模拟真实环境中的高噪声数据 (50% 噪声)
        noise = np.random.normal(0, FIXED_NOISE * y_train_clean_sub)
        y_train_noisy = np.maximum(y_train_clean_sub + noise, 1e-6)
        
        # C. 训练 MLP 去噪器 (MLP Denoiser)
        scaler_X = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train_sub)
        
        mlp = MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu',
            learning_rate_init=0.001,
            max_iter=500, 
            random_state=seed
        )
        try:
            mlp.fit(X_train_scaled, y_train_noisy)
        except Exception as e:
            print(f"    [Warning] MLP fit failed: {e}")
            continue

        # D. PySR 混合训练 (Hybrid PySR Training)
        # 为了速度，PySR 的输入样本量上限限制在 2000
        pysr_n = min(current_size, 2000)
        pysr_idx = np.random.choice(len(X_train_sub), pysr_n, replace=False)
        
        X_pysr_train = X_train_sub[pysr_idx]
        X_pysr_train_scaled = X_train_scaled[pysr_idx]
        
        # 核心步骤：使用 MLP 对 PySR 的训练数据进行"预清洗"
        y_denoised = mlp.predict(X_pysr_train_scaled)
        y_denoised = np.maximum(y_denoised, 1e-6)
        
        # PySR 拟合去噪后的数据
        model_hybrid = PySRRegressor(
            niterations=40, # 迭代次数较少，追求快速验证
            populations=15,
            binary_operators=["+", "*", "-", "/"],
            unary_operators=["inv(x)=1/x", "sqrt", "square"],
            extra_sympy_mappings={'inv': lambda x: 1/x},
            verbosity=0,
            temp_equation_file=True,
            delete_tempfiles=True 
        )
        
        r2 = -1.0 # 默认失败分数
        try:
            model_hybrid.fit(X_pysr_train, y_denoised, variable_names=FEATURE_NAMES)
            
            # --- 安全性检查 ---
            y_pred = model_hybrid.predict(X_test)
            
            # 检查是否有无穷大或NaN (可能是生成的公式有奇点)
            if not np.all(np.isfinite(y_pred)):
                print(f"    Repeat {i+1}/{N_REPEATS} | [Warning] PySR 生成了奇点公式 (Inf/NaN)，本次跳过。")
                r2 = 0.0 
            else:
                # 在干净的测试集上评估 R2
                r2 = r2_score(y_test_clean, y_pred)
                print(f"    Repeat {i+1}/{N_REPEATS} | Hybrid R2: {r2:.4f}")
                
        except Exception as e:
            print(f"    Repeat {i+1}/{N_REPEATS} | [Error] PySR Predict Error: {e}")
            r2 = 0.0

        r2_scores_hybrid.append(r2)

    # E. 统计结果 (Statistics)
    valid_scores = [s for s in r2_scores_hybrid if s > 0.0]
    if len(valid_scores) > 0:
        mean_r2 = np.mean(valid_scores)
        std_r2 = np.std(valid_scores)
    else:
        mean_r2 = 0.0
        std_r2 = 0.0
    
    print(f"  >>> Size {current_size} 结果: Mean R2 = {mean_r2:.4f} (Std: {std_r2:.4f})")
    
    # 保存该数据规模下的统计结果
    results_summary.append({
        'Training_Size': current_size,
        'Mean_R2': mean_r2,
        'Std_R2': std_r2,
        'Raw_Scores': str(r2_scores_hybrid)
    })
    
    pd.DataFrame(results_summary).to_csv(os.path.join(OUTPUT_DIR, "Data_Efficiency_Curve.csv"), index=False)

print("\n" + "="*50)
print("实验运行完成。数据已保存至 Data_Efficiency_Curve.csv")
print("="*50)

正在加载数据...
数据准备完毕。实验将重复 20 次。

正在评估训练集大小: 100


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    Repeat 1/20 | Hybrid R2: 0.9286


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    Repeat 2/20 | Hybrid R2: 0.9002


...

    Repeat 15/20 | Hybrid R2: 0.9804


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    Repeat 16/20 | Hybrid R2: 0.9828


D:\Users\A\anaconda3\envs\py310_llm\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(


    Repeat 17/20 | Hybrid R2: 0.9836
